# Упражнение: Point-in-time correct join без утечки из будущего

## Задание

Соберите обучающую выборку так, как это делает `get_historical_features` в feature store.

- `input.csv` — метки: `label_id, user_id, ts, churned` (момент, на который известна метка);
- `orders_features.csv` — история значений признака `orders_30d`: `user_id, ts, orders_30d`;
- `tickets_features.csv` — история признака `tickets_90d`: `user_id, ts, tickets_90d`.

Правила (одинаковые для обучения и для онлайн-сервиса):

1. Для каждой метки берётся **последнее значение признака того же пользователя с `ts` строго раньше**
   момента метки. Значение, записанное в тот же день, считается ещё недоступным (пакетный расчёт
   завершается в конце дня).
2. У `orders_30d` есть **TTL = 30 дней**: если найденное значение старше 30 дней
   (разница дат > 30), оно считается устаревшим и не используется. У `tickets_90d` TTL нет.
3. Если значения нет (или оно устарело) — запишите `-1`.
4. `orders_age_days` — возраст использованного значения `orders_30d` в днях (`-1`, если значения нет).
5. `leak` = 1, если «наивный» join — подставить пользователю **самое последнее** известное значение
   `orders_30d` из всей истории — дал бы другое число, чем point-in-time join (в том числе когда PIT даёт `-1`).

## Подсказки

- Посмотрите на пример с `pd.merge_asof` в лекции: обе таблицы нужно отсортировать по `ts`,
  а соединять — `on='ts', by='user_id', direction='backward'`.
- «Строго раньше» — параметр `allow_exact_matches=False`; TTL — `tolerance=pd.Timedelta('30D')`.
  Чтобы посчитать возраст значения, сохраните его дату отдельным столбцом до merge.
- `merge_asof` меняет порядок строк — верните его сортировкой по `label_id`. Для проверки напишите
  медленный вариант циклом: для каждой метки отфильтруйте историю пользователя `ts < label_ts` и возьмите последнюю строку.

## Ожидаемый формат выходного файла

CSV с колонками `label_id, user_id, orders_30d, orders_age_days, tickets_90d, leak` —
16 строк по возрастанию `label_id`; все значения целые.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/feature_stores
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.